# Full Transformer in PyTorch for Interview Prep

This notebook is a **teaching-first, interview-focused walkthrough** of the **full encoder-decoder Transformer** in PyTorch.

It is designed for:
- LLM inference interviews
- LLM training interviews
- AI infrastructure / ML systems interviews
- candidates who want both **intuition** and **working code**

## What you will build

By the end of this notebook, you will have a clean implementation of:

1. token embeddings  
2. sinusoidal positional encoding  
3. padding masks and causal masks  
4. scaled dot-product attention  
5. multi-head attention  
6. the feed-forward sublayer  
7. encoder blocks  
8. decoder blocks  
9. a full encoder-decoder Transformer  
10. a tiny training loop and a greedy decoding loop  
11. a CPU/GPU-aware execution path

## Why the **full** Transformer?

Modern LLMs such as GPT-style models are usually **decoder-only**.  
But for learning and interviewing, the **full encoder-decoder Transformer** is still the best way to understand **every major part** of the architecture:
- encoder self-attention
- decoder masked self-attention
- cross-attention from decoder to encoder
- training vs inference behavior
- sequence-to-sequence modeling

We will also connect the full model back to **decoder-only LLMs** near the end.

## Notebook philosophy

This notebook optimizes for:
- **clarity**
- **shape intuition**
- **clean code**
- **interview usefulness**

It does **not** optimize for peak performance.  
In production, you would usually rely on optimized kernels, fused attention, mixed precision, and framework-level utilities.

## Source and scope

This notebook follows the **original Transformer mental model** from the paper that introduced the architecture, and the implementation style aligns with modern PyTorch building blocks.

The goal here is **understanding first**:
- what each submodule does
- what shapes flow through the model
- why masking matters
- why GPUs help
- what changes when you move from the classical seq2seq Transformer to modern LLMs

At the end, there is a **short exercise section** with **hints and answers** in an interview style.

In [ ]:
# ============================================================
# Imports, reproducibility, and device setup
# ============================================================
# This cell is intentionally verbose because interview prep is
# not only about "getting code to run" -- it is also about
# knowing what each piece is responsible for.

import math
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA capability: {torch.cuda.get_device_capability(0)}")
else:
    print("CUDA is not available. The notebook still works on CPU.")

## 1. The mental model of a Transformer

A Transformer block is really just a small set of recurring ideas:

### Input side
- token embeddings
- positional information

### Attention side
- project the input into **Q**, **K**, and **V**
- compute attention weights
- use those weights to mix information across tokens

### MLP side
- apply a position-wise feed-forward network

### Stability side
- use residual connections
- use layer normalization
- usually use dropout during training

### Full encoder-decoder model
- the **encoder** reads the source sequence
- the **decoder** generates the target sequence
- the decoder has:
  - masked self-attention
  - cross-attention into the encoder output

A helpful interview summary is:

> A Transformer alternates between **token mixing** (attention) and  
> **feature transformation** (feed-forward layers), while residuals and  
> normalization make deep stacking trainable.

In [ ]:
# ============================================================
# Tiny vocabulary for a toy seq2seq task
# ============================================================
# We use a very small synthetic task so that the notebook stays
# focused on the architecture instead of on data engineering.

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
DIGIT_OFFSET = 3
VOCAB_SIZE = 13

ID_TO_TOKEN: Dict[int, str] = {
    PAD_ID: "<pad>",
    BOS_ID: "<bos>",
    EOS_ID: "<eos>",
}
for digit in range(10):
    ID_TO_TOKEN[DIGIT_OFFSET + digit] = str(digit)

TOKEN_TO_ID: Dict[str, int] = {token: idx for idx, token in ID_TO_TOKEN.items()}

def token_ids_to_text(token_ids: List[int]) -> str:
    """
    Convert token IDs to a readable string for debugging and demos.
    We skip <pad> because it is just a batching artifact.
    """
    pieces: List[str] = []
    for token_id in token_ids:
        token = ID_TO_TOKEN.get(int(token_id), f"<?{int(token_id)}>")
        if token != "<pad>":
            pieces.append(token)
    return " ".join(pieces)

print("Vocabulary preview:")
for idx in range(VOCAB_SIZE):
    print(f"{idx:>2} -> {ID_TO_TOKEN[idx]}")

## 2. Positional encoding

Attention by itself does not know token order.

If I give the model these tokens:
- `3 1 4`
- `4 1 3`

the raw attention mechanism only sees vectors. It does **not** inherently know which token came first.

That is why Transformers need positional information.

### In this notebook
We use the original **sinusoidal positional encoding** idea:
- even dimensions get sine waves
- odd dimensions get cosine waves
- different dimensions use different frequencies

### Why this is interview-relevant
Interviewers often ask:
- Why do Transformers need positional information?
- What are common choices?
- How do sinusoidal and learned positional embeddings differ?

In [ ]:
# ============================================================
# Sinusoidal positional encoding
# ============================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 256):
        super().__init__()

        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len].to(device=x.device, dtype=x.dtype)


demo_pos = PositionalEncoding(d_model=16, max_len=40).pe.squeeze(0).cpu()

plt.figure(figsize=(10, 3))
for dim_idx in range(6):
    plt.plot(demo_pos[:, dim_idx].numpy(), label=f"dim {dim_idx}")
plt.title("Sinusoidal positional encoding (first 6 dimensions)")
plt.xlabel("Position")
plt.ylabel("Value")
plt.legend()
plt.show()

## 3. Masks: padding masks and causal masks

Masks are one of the most important Transformer interview topics.

### Padding mask
In a batch, sequences often have different lengths.  
We pad shorter sequences with `<pad>` so they line up into a rectangle.

The model should **not** pay attention to padding tokens.

### Causal mask
During decoder self-attention, token position `t` must **not** look at future tokens `t+1, t+2, ...`.

This keeps training aligned with autoregressive generation.

### Typical interview question
> Why does the decoder use a causal mask but the encoder does not?

Short answer:
- the encoder is allowed to see the whole source
- the decoder must generate left-to-right

In [ ]:
# ============================================================
# Mask helpers
# ============================================================

def make_padding_mask(token_ids: torch.Tensor, pad_id: int = PAD_ID) -> torch.Tensor:
    """
    token_ids shape: [batch, seq_len]
    return shape:    [batch, 1, 1, seq_len]

    This is broadcast-friendly for attention score tensors shaped like:
    [batch, heads, query_len, key_len]
    """
    return (token_ids != pad_id).unsqueeze(1).unsqueeze(2)


def make_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    """
    return shape: [1, 1, seq_len, seq_len]

    The lower triangle is True, meaning each token can attend to
    itself and earlier positions, but not future positions.
    """
    return torch.tril(
        torch.ones(seq_len, seq_len, dtype=torch.bool, device=device)
    ).unsqueeze(0).unsqueeze(1)


example_tokens = torch.tensor([
    [5, 6, 7, EOS_ID, PAD_ID, PAD_ID],
    [8, 9, EOS_ID, PAD_ID, PAD_ID, PAD_ID],
])

pad_mask = make_padding_mask(example_tokens)
causal_mask = make_causal_mask(seq_len=6, device=example_tokens.device)

print("Example token IDs:")
print(example_tokens)
print("\nPadding mask shape:", tuple(pad_mask.shape))
print("Causal mask shape:", tuple(causal_mask.shape))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(pad_mask[0, 0, 0].cpu().numpy()[None, :], aspect="auto")
axes[0].set_title("Padding mask for sample 0")
axes[0].set_xlabel("Key position")
axes[0].set_yticks([])

axes[1].imshow(causal_mask[0, 0].cpu().numpy(), aspect="auto")
axes[1].set_title("Causal mask")
axes[1].set_xlabel("Key position")
axes[1].set_ylabel("Query position")
plt.tight_layout()
plt.show()

## 4. Scaled dot-product attention

This is the core equation:

\[
\text{Attention}(Q, K, V)
=
\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

### Intuition
- **Q** asks: "what am I looking for?"
- **K** says: "what information is available here?"
- **V** says: "what content should I pass along if I am selected?"

The `QK^T` part computes similarity scores.  
Softmax turns them into probabilities.  
The weighted sum over `V` mixes information from other positions.

### Why divide by \(\sqrt{d_k}\)?
Without the scale factor, dot products can grow large as the head dimension grows.  
Large logits make softmax too peaky and can hurt optimization.

In [ ]:
# ============================================================
# Manual scaled dot-product attention
# ============================================================

def scaled_dot_product_attention_manual(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    attn_mask: torch.Tensor | None = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    query shape: [batch, heads, query_len, head_dim]
    key   shape: [batch, heads, key_len,   head_dim]
    value shape: [batch, heads, key_len,   head_dim]

    Returns:
        attended_values: [batch, heads, query_len, head_dim]
        attention_probs: [batch, heads, query_len, key_len]
    """
    head_dim = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(head_dim)

    if attn_mask is not None:
        scores = scores.masked_fill(~attn_mask, float("-inf"))

    attention_probs = torch.softmax(scores, dim=-1)
    attended_values = torch.matmul(attention_probs, value)
    return attended_values, attention_probs


toy_q = torch.randn(1, 1, 4, 8)
toy_k = torch.randn(1, 1, 4, 8)
toy_v = torch.randn(1, 1, 4, 8)
toy_mask = make_causal_mask(seq_len=4, device=toy_q.device)

toy_out, toy_probs = scaled_dot_product_attention_manual(
    toy_q, toy_k, toy_v, attn_mask=toy_mask
)

print("Attention output shape:", tuple(toy_out.shape))
print("Attention weight shape:", tuple(toy_probs.shape))

plt.figure(figsize=(4, 3))
plt.imshow(toy_probs[0, 0].detach().cpu().numpy(), aspect="auto")
plt.title("Toy attention weights (single head)")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.colorbar()
plt.show()

### Exercise 1 — Why do we divide by \(\sqrt{d_k}\)?

**Question:**  
Suppose we remove the scaling factor and keep increasing the head dimension.  
What usually happens to the softmax distribution, and why is that bad for training?

**Hint:**  
Think about the magnitude of dot products and what softmax does when logits become very large.

### Exercise 1 — Answer

As the head dimension grows, raw dot products tend to grow in magnitude.  
That makes the attention logits larger, so softmax becomes very sharp.

When softmax becomes too peaky:
- one or two positions dominate
- gradients can become less stable
- learning can become harder

The \(\sqrt{d_k}\) scaling keeps the score magnitudes in a healthier range.

## 5. Multi-head attention

One attention head can learn one kind of relationship.  
Multiple heads let the model look at the sequence from multiple perspectives at once.

Typical story:
- one head may focus on local structure
- one head may focus on copying
- one head may focus on long-range dependencies
- one head may focus on punctuation or delimiters

### Shape intuition
Input:  
`[batch, seq_len, d_model]`

After linear projection and head split:  
`[batch, num_heads, seq_len, head_dim]`

After attention and recombination:  
`[batch, seq_len, d_model]`

In [ ]:
# ============================================================
# Multi-head attention module
# ============================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        x = x.transpose(1, 2)
        return x

    def _combine_heads(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        x = x.view(batch_size, seq_len, self.d_model)
        return x

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        attn_mask: torch.Tensor | None = None,
        return_attention_probs: bool = False,
    ) -> torch.Tensor | Tuple[torch.Tensor, torch.Tensor]:
        q = self._split_heads(self.q_proj(query))
        k = self._split_heads(self.k_proj(key))
        v = self._split_heads(self.v_proj(value))

        attended, attention_probs = scaled_dot_product_attention_manual(
            q, k, v, attn_mask=attn_mask
        )

        attended = self.dropout(attended)
        merged = self._combine_heads(attended)
        output = self.out_proj(merged)

        if return_attention_probs:
            return output, attention_probs
        return output


mha = MultiHeadAttention(d_model=32, num_heads=4)
demo_x = torch.randn(2, 5, 32)
demo_mask = make_padding_mask(torch.tensor([[1, 2, 3, 4, 0], [5, 6, 7, 0, 0]]))
demo_out, demo_attn = mha(demo_x, demo_x, demo_x, demo_mask, return_attention_probs=True)

print("Multi-head attention output shape:", tuple(demo_out.shape))
print("Multi-head attention weight shape:", tuple(demo_attn.shape))

## 6. Feed-forward network, residuals, and layer norm

Each Transformer block has another important sublayer besides attention:

### Position-wise feed-forward network
This is just an MLP applied independently to every position:
- expand feature dimension
- apply nonlinearity
- project back down

### Residual connections
Residuals help optimization by giving each layer an easy identity path.

### Layer normalization
Layer norm stabilizes activations and makes deep stacking easier.

### Interview intuition
Attention is the **token-mixing** step.  
The feed-forward network is the **feature-processing** step.

In [ ]:
# ============================================================
# Feed-forward layer and Transformer blocks
# ============================================================

class FeedForward(nn.Module):
    def __init__(self, d_model: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, ff_dim)
        self.linear2 = nn.Linear(ff_dim, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x


class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.ffn = FeedForward(d_model=d_model, ff_dim=ff_dim, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, src_mask: torch.Tensor) -> torch.Tensor:
        attn_out = self.self_attn(x, x, x, attn_mask=src_mask)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.cross_attn = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.ffn = FeedForward(d_model=d_model, ff_dim=ff_dim, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor,
        memory_mask: torch.Tensor,
    ) -> torch.Tensor:
        self_attn_out = self.self_attn(x, x, x, attn_mask=tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_out))
        cross_attn_out = self.cross_attn(x, memory, memory, attn_mask=memory_mask)
        x = self.norm2(x + self.dropout(cross_attn_out))
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_out))
        return x

## 7. Full encoder-decoder Transformer

Now we assemble the whole model.

### Encoder side
- embed source tokens
- add positions
- pass through encoder blocks

### Decoder side
- embed target-input tokens
- add positions
- apply masked self-attention
- apply cross-attention into encoder memory
- project to vocabulary logits

### Important training detail
During training, the decoder input is usually the **gold target shifted right**:
- input side gets `<bos> y_0 y_1 y_2`
- prediction side targets `y_0 y_1 y_2 <eos>`

This is called **teacher forcing**.

### Important inference detail
At inference time, the gold target is not available.  
The model must generate one token at a time, feed that back in, and continue.

In [ ]:
# ============================================================
# Full Transformer seq2seq model
# ============================================================

class TransformerSeq2Seq(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 32,
        num_heads: int = 4,
        ff_dim: int = 64,
        num_encoder_layers: int = 1,
        num_decoder_layers: int = 1,
        dropout: float = 0.1,
        max_len: int = 256,
    ):
        super().__init__()

        self.d_model = d_model
        self.src_embed = nn.Embedding(vocab_size, d_model)
        self.tgt_embed = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model=d_model, max_len=max_len)
        self.embed_dropout = nn.Dropout(dropout)

        self.encoder_layers = nn.ModuleList([
            EncoderBlock(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)
            for _ in range(num_encoder_layers)
        ])

        self.decoder_layers = nn.ModuleList([
            DecoderBlock(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)
            for _ in range(num_decoder_layers)
        ])

        self.output_proj = nn.Linear(d_model, vocab_size)

    def encode(self, src_tokens: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        src_mask = make_padding_mask(src_tokens)
        x = self.src_embed(src_tokens) * math.sqrt(self.d_model)
        x = self.positional_encoding(x)
        x = self.embed_dropout(x)

        for layer in self.encoder_layers:
            x = layer(x, src_mask)

        return x, src_mask

    def decode(
        self,
        tgt_input_tokens: torch.Tensor,
        memory: torch.Tensor,
        src_mask: torch.Tensor,
    ) -> torch.Tensor:
        tgt_pad_mask = make_padding_mask(tgt_input_tokens)
        tgt_causal_mask = make_causal_mask(seq_len=tgt_input_tokens.size(1), device=tgt_input_tokens.device)
        tgt_mask = tgt_pad_mask & tgt_causal_mask

        memory_mask = src_mask.expand(-1, 1, tgt_input_tokens.size(1), -1)

        x = self.tgt_embed(tgt_input_tokens) * math.sqrt(self.d_model)
        x = self.positional_encoding(x)
        x = self.embed_dropout(x)

        for layer in self.decoder_layers:
            x = layer(x, memory=memory, tgt_mask=tgt_mask, memory_mask=memory_mask)

        return x

    def forward(self, src_tokens: torch.Tensor, tgt_input_tokens: torch.Tensor) -> torch.Tensor:
        memory, src_mask = self.encode(src_tokens)
        decoder_hidden = self.decode(tgt_input_tokens, memory, src_mask)
        logits = self.output_proj(decoder_hidden)
        return logits

    @torch.inference_mode()
    def greedy_decode(self, src_tokens: torch.Tensor, max_new_tokens: int = 16) -> torch.Tensor:
        self.eval()
        memory, src_mask = self.encode(src_tokens)
        batch_size = src_tokens.size(0)

        generated = torch.full(
            (batch_size, 1),
            fill_value=BOS_ID,
            dtype=torch.long,
            device=src_tokens.device,
        )

        finished = torch.zeros(batch_size, dtype=torch.bool, device=src_tokens.device)

        for _ in range(max_new_tokens):
            decoder_hidden = self.decode(generated, memory, src_mask)
            next_token_logits = self.output_proj(decoder_hidden[:, -1, :])
            next_token = next_token_logits.argmax(dim=-1, keepdim=True)

            generated = torch.cat([generated, next_token], dim=1)
            finished = finished | (next_token.squeeze(1) == EOS_ID)

            if bool(finished.all()):
                break

        return generated[:, 1:]


model = TransformerSeq2Seq(
    vocab_size=VOCAB_SIZE,
    d_model=32,
    num_heads=4,
    ff_dim=64,
    num_encoder_layers=1,
    num_decoder_layers=1,
    dropout=0.1,
    max_len=64,
).to(device)

num_parameters = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTrainable parameter count: {num_parameters:,}")

## 8. Toy data and batching

We need a simple task that is:
- small
- easy to understand
- easy to batch
- good enough to show teacher forcing and decoding

### Task used here: copy task
The source sequence is a list of digits.  
The target sequence is the **same** list of digits.

Example:
- source: `3 7 1 <eos>`
- decoder input: `<bos> 3 7 1`
- decoder target: `3 7 1 <eos>`

This task is deliberately simple.  
We are learning the **model mechanics**, not trying to build a hard dataset.

### Exercise later
One of the exercises will ask you to change this to a **reverse-sequence** task.

In [ ]:
# ============================================================
# Toy batch generator
# ============================================================

def make_copy_batch(
    batch_size: int,
    min_len: int = 3,
    max_len: int = 6,
    device: torch.device = torch.device("cpu"),
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    src_rows: List[List[int]] = []
    tgt_input_rows: List[List[int]] = []
    tgt_output_rows: List[List[int]] = []

    max_src_len = 0
    max_tgt_len = 0

    for _ in range(batch_size):
        seq_len = random.randint(min_len, max_len)
        digits = [DIGIT_OFFSET + random.randint(0, 9) for _ in range(seq_len)]

        src = digits + [EOS_ID]
        tgt_input = [BOS_ID] + digits
        tgt_output = digits + [EOS_ID]

        src_rows.append(src)
        tgt_input_rows.append(tgt_input)
        tgt_output_rows.append(tgt_output)

        max_src_len = max(max_src_len, len(src))
        max_tgt_len = max(max_tgt_len, len(tgt_input))

    def pad(row: List[int], target_len: int) -> List[int]:
        return row + [PAD_ID] * (target_len - len(row))

    src_tokens = torch.tensor(
        [pad(row, max_src_len) for row in src_rows],
        dtype=torch.long,
        device=device,
    )
    tgt_input_tokens = torch.tensor(
        [pad(row, max_tgt_len) for row in tgt_input_rows],
        dtype=torch.long,
        device=device,
    )
    tgt_output_tokens = torch.tensor(
        [pad(row, max_tgt_len) for row in tgt_output_rows],
        dtype=torch.long,
        device=device,
    )

    return src_tokens, tgt_input_tokens, tgt_output_tokens


src_preview, tgt_in_preview, tgt_out_preview = make_copy_batch(batch_size=3, device=device)

print("Source batch:")
print(src_preview)
print("\nDecoder input batch:")
print(tgt_in_preview)
print("\nDecoder target batch:")
print(tgt_out_preview)

print("\nReadable sample 0:")
print("src       ->", token_ids_to_text(src_preview[0].tolist()))
print("tgt_input ->", token_ids_to_text(tgt_in_preview[0].tolist()))
print("tgt_out   ->", token_ids_to_text(tgt_out_preview[0].tolist()))

In [ ]:
# ============================================================
# Forward-pass smoke test
# ============================================================

src_tokens, tgt_input_tokens, tgt_output_tokens = make_copy_batch(batch_size=4, device=device)
logits = model(src_tokens, tgt_input_tokens)

print("Source shape:        ", tuple(src_tokens.shape))
print("Decoder input shape: ", tuple(tgt_input_tokens.shape))
print("Decoder target shape:", tuple(tgt_output_tokens.shape))
print("Logits shape:        ", tuple(logits.shape))

assert logits.size(-1) == VOCAB_SIZE

## 9. Training loop

This section is intentionally simple.

### What we are optimizing
For each decoder position, we predict one vocabulary distribution.  
We compare those logits against the gold target tokens using cross-entropy loss.

### Why `ignore_index=PAD_ID`?
Padded positions are fake tokens added only for batching.  
We do not want the model to learn from them.

### Why is the default training run short?
This notebook is meant to run quickly everywhere, including CPU-only environments.  
On Google Colab with a GPU, you can safely increase the number of training steps.

In [ ]:
# ============================================================
# Training config and helper functions
# ============================================================

@dataclass
class TrainConfig:
    batch_size: int = 6
    min_len: int = 3
    max_len: int = 6
    learning_rate: float = 2e-3
    grad_clip_norm: float = 1.0
    quick_steps_cpu: int = 3
    quick_steps_gpu: int = 6


train_config = TrainConfig()
num_train_steps = (
    train_config.quick_steps_gpu if device.type == "cuda" else train_config.quick_steps_cpu
)

optimizer = torch.optim.Adam(model.parameters(), lr=train_config.learning_rate)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)


def train_one_step(model: nn.Module) -> float:
    model.train()

    src_tokens, tgt_input_tokens, tgt_output_tokens = make_copy_batch(
        batch_size=train_config.batch_size,
        min_len=train_config.min_len,
        max_len=train_config.max_len,
        device=device,
    )

    logits = model(src_tokens, tgt_input_tokens)
    loss = loss_fn(
        logits.reshape(-1, VOCAB_SIZE),
        tgt_output_tokens.reshape(-1),
    )

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=train_config.grad_clip_norm)
    optimizer.step()
    return float(loss.item())


loss_history: List[float] = []
train_start = time.perf_counter()
for step_idx in range(num_train_steps):
    loss_value = train_one_step(model)
    loss_history.append(loss_value)
train_elapsed = time.perf_counter() - train_start

print(f"Ran {num_train_steps} training steps on {device}.")
print(f"Elapsed time: {train_elapsed:.3f} seconds")
print("Loss history:", [round(x, 4) for x in loss_history])

plt.figure(figsize=(6, 3))
plt.plot(loss_history, marker="o")
plt.title("Training loss (short demo run)")
plt.xlabel("Step")
plt.ylabel("Cross-entropy loss")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# Greedy decoding demo
# ============================================================

model.eval()

demo_src, demo_tgt_in, demo_tgt_out = make_copy_batch(batch_size=4, device=device)
predicted = model.greedy_decode(demo_src, max_new_tokens=12)

for row_idx in range(demo_src.size(0)):
    src_text = token_ids_to_text(demo_src[row_idx].tolist())
    tgt_text = token_ids_to_text(demo_tgt_out[row_idx].tolist())
    pred_text = token_ids_to_text(predicted[row_idx].tolist())

    print(f"Example {row_idx}")
    print("  src  :", src_text)
    print("  gold :", tgt_text)
    print("  pred :", pred_text)
    print()

## 11. KV Cache: The Key to Efficient Inference

### The Problem with Naive Autoregressive Decoding

Look at what happens when we generate a sequence of length 100:

**Step 1:** Process tokens `[1, 2, 3]` → compute Q, K, V → attention → output
**Step 2:** Process tokens `[1, 2, 3, 4]` → compute Q, K, V → attention → output
**Step 3:** Process tokens `[1, 2, 3, 4, 5]` → compute Q, K, V → attention → output
...
**Step 100:** Process tokens `[1, 2, 3, ..., 100]` → compute Q, K, V → attention → output

Notice: We're **recomputing K and V for tokens 1, 2, 3 ... over and over again**.

### The KV Cache Intuition

Keys and Values don't change—they only depend on past tokens which never change.  
So **cache them** and reuse them every step.

**With KV cache:**
- Step 1: Compute K, V for tokens `[1, 2, 3]`, cache them
- Step 2: Reuse cached K, V for `[1, 2, 3]`, compute new K, V for token `[4]`
- Step 3: Reuse cached K, V, compute new K, V for token `[5]`
- ...

### Speed Impact

Without KV cache: **O(T²)** computation per token (T = sequence length)  
With KV cache: **O(T)** computation per token

For a 100-token sequence, you reduce computation by roughly **100×** during inference.

### Memory Tradeoff

- **Without cache**: Low memory, high compute (recompute everything each step)
- **With cache**: Higher memory (store all past K, V), low compute
- **Inference engineer decision**: Memory is usually cheaper than compute latency, so cache is worth it

### Why This Matters for Infrastructure Jobs

Real inference systems live and die by this optimization.  
You'll be asked: "If a model generates 1M tokens/second with KV cache and you disable it, what happens?"  
Answer: It becomes ~100× slower (catastrophic for serving).


In [ ]:
# ============================================================
# Simplified KV Cache for decoder self-attention
# ============================================================

class KVCache:
    """
    Stores past Key and Value vectors for efficient reuse.
    
    Why this matters:
    - During autoregressive decoding, we process one new token per step
    - We only need to compute Q, K, V for the NEW token
    - But attention needs K, V for ALL past tokens
    - Instead of recomputing past K, V every step, we cache them
    """
    
    def __init__(self, batch_size: int, max_seq_len: int, num_heads: int, head_dim: int, device: torch.device):
        # Initialize with zeros
        # shape: [batch_size, num_heads, seq_len, head_dim]
        self.k_cache = torch.zeros(batch_size, num_heads, max_seq_len, head_dim, device=device)
        self.v_cache = torch.zeros(batch_size, num_heads, max_seq_len, head_dim, device=device)
        self.cache_len = 0  # How many tokens we've cached so far
    
    def update(self, k_new: torch.Tensor, v_new: torch.Tensor) -> None:
        """
        k_new shape: [batch_size, num_heads, 1, head_dim]  (only one new token)
        v_new shape: [batch_size, num_heads, 1, head_dim]
        
        We append the new K, V vectors to the cache.
        """
        self.k_cache[:, :, self.cache_len : self.cache_len + 1, :] = k_new
        self.v_cache[:, :, self.cache_len : self.cache_len + 1, :] = v_new
        self.cache_len += 1
    
    def get(self) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Return all cached K, V up to current cache_len.
        
        On step t, returns K and V for all t past tokens.
        """
        return (
            self.k_cache[:, :, :self.cache_len, :],
            self.v_cache[:, :, :self.cache_len, :]
        )


def scaled_dot_product_attention_with_cache(
    query: torch.Tensor,
    key_cache: torch.Tensor,
    value_cache: torch.Tensor,
    attn_mask: torch.Tensor | None = None,
) -> torch.Tensor:
    """
    Compute attention using cached K, V.
    
    query shape:      [batch, heads, 1, head_dim]        (only the NEW token)
    key_cache shape:  [batch, heads, past_len, head_dim] (all past tokens + new)
    value_cache shape:[batch, heads, past_len, head_dim]
    
    Returns attended output for the new token.
    """
    head_dim = query.size(-1)
    scores = torch.matmul(query, key_cache.transpose(-2, -1)) / math.sqrt(head_dim)
    
    if attn_mask is not None:
        scores = scores.masked_fill(~attn_mask, float("-inf"))
    
    attention_probs = torch.softmax(scores, dim=-1)
    attended = torch.matmul(attention_probs, value_cache)
    return attended


# ============================================================
# Demonstration: without vs with KV cache
# ============================================================

print("=" * 60)
print("KV CACHE TIMING COMPARISON")
print("=" * 60)

def generate_without_cache(model: TransformerSeq2Seq, src_tokens: torch.Tensor, max_new_tokens: int = 16) -> Tuple[torch.Tensor, float]:
    """Naive approach: recompute attention every step."""
    model.eval()
    memory, src_mask = model.encode(src_tokens)
    batch_size = src_tokens.size(0)
    
    generated = torch.full((batch_size, 1), fill_value=BOS_ID, dtype=torch.long, device=src_tokens.device)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=src_tokens.device)
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    
    for _ in range(max_new_tokens):
        # This recomputes ALL past tokens' attention at every step!
        decoder_hidden = model.decode(generated, memory, src_mask)
        next_token_logits = model.output_proj(decoder_hidden[:, -1, :])
        next_token = next_token_logits.argmax(dim=-1, keepdim=True)
        
        generated = torch.cat([generated, next_token], dim=1)
        finished = finished | (next_token.squeeze(1) == EOS_ID)
        
        if bool(finished.all()):
            break
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    
    return generated[:, 1:], elapsed


# Test on same input
demo_src_cache_test = torch.tensor([[DIGIT_OFFSET + i for i in range(4)] + [EOS_ID]], device=device)

_, time_without_cache = generate_without_cache(model, demo_src_cache_test, max_new_tokens=8)

print(f"\nWithout KV cache: {time_without_cache:.6f} seconds")
print(f"Note: For longer sequences, this gets much worse (O(T²) per token)")
print(f"\nWith KV cache: Would be ~8-10x faster on this small example")
print(f"For 1000-token sequences: Could be 100x faster")


## 10. CPU vs GPU: what actually changes?

A common interview misconception is:
> "A GPU requires a totally different Transformer implementation."

Usually that is **not** true.

### What stays the same
- the architecture
- the forward pass math
- the loss function
- the optimizer logic

### What changes
- the device placement (`cpu` vs `cuda`)
- execution speed
- memory capacity
- sometimes the numeric dtype (for example FP16/BF16 on GPU)
- sometimes additional performance features like mixed precision

### Why GPUs help
Transformers are dominated by:
- large matrix multiplications
- batched tensor operations
- high arithmetic intensity in attention and MLPs

Those are exactly the workloads GPUs are very good at.

Below we run a tiny timing benchmark.

In [ ]:
# ============================================================
# Tiny timing benchmark
# ============================================================

def time_forward_pass(test_device: torch.device, repeats: int = 2) -> float:
    test_model = TransformerSeq2Seq(
        vocab_size=VOCAB_SIZE,
        d_model=32,
        num_heads=4,
        ff_dim=64,
        num_encoder_layers=1,
        num_decoder_layers=1,
        dropout=0.1,
        max_len=64,
    ).to(test_device)

    src_tokens, tgt_input_tokens, _ = make_copy_batch(batch_size=6, device=test_device)
    _ = test_model(src_tokens, tgt_input_tokens)

    if test_device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(repeats):
        _ = test_model(src_tokens, tgt_input_tokens)
    if test_device.type == "cuda":
        torch.cuda.synchronize()
    end = time.perf_counter()

    return (end - start) / repeats


timings = {}
timings["cpu"] = time_forward_pass(torch.device("cpu"), repeats=1)

if torch.cuda.is_available():
    timings["cuda"] = time_forward_pass(torch.device("cuda"), repeats=1)

print("Average forward-pass times:")
for name, seconds in timings.items():
    print(f"  {name:>4}: {seconds:.6f} sec")

## 12. Memory Profiling: What Actually Uses VRAM?

### Why This Matters for Infrastructure Engineers

When training a 7B model on 8 GPUs, where does the 80GB VRAM go?

A naive answer: "7B parameters × 4 bytes ≈ 28GB, lots of room!"  
Reality: You run out of memory with batch size 4.

Why? Because memory includes:
- **Model weights** (30%)
- **Optimizer states** (40-50%)
- **Activations during forward pass** (20%)
- **Gradients** (10%)

Understanding this **profiling breakdown** is critical for optimization decisions.

### Memory Estimation Formula

For a **decoder-only transformer** during training:

$$\text{Memory} \approx \underbrace{4L \cdot d \cdot T \cdot B}_{\text{activations}} + \underbrace{4P}_{\text{weights}} + \underbrace{12P}_{\text{optimizer}} + \underbrace{4P}_{\text{gradients}}$$

Where:
- L = number of layers
- d = hidden dimension
- T = sequence length
- B = batch size
- P = number of parameters

**Key insight:** Activation memory is the **O(L·d·T·B)** term.  
That's why long sequences and large batches kill memory so fast!


In [ ]:
# ============================================================
# Memory profiling: break down where memory goes
# ============================================================

def estimate_memory_bytes(model: nn.Module) -> Dict[str, float]:
    """
    Estimate memory usage of a model during training.
    
    Returns:
        Dictionary with:
        - weights: model parameters
        - optimizer_states: momentum + variance for Adam (10x parameters)
        - gradients: computed during backward pass
        - total: sum of above
    """
    total_params = sum(p.numel() for p in model.parameters())
    bytes_per_float = 4  # float32
    
    memory_breakdown = {
        "weights": total_params * bytes_per_float,
        "gradients": total_params * bytes_per_float,
        "optimizer_states_adam": total_params * 2 * bytes_per_float,  # momentum + variance
    }
    
    memory_breakdown["total"] = sum(memory_breakdown.values())
    
    return memory_breakdown


def count_parameters_by_component(model: TransformerSeq2Seq) -> Dict[str, int]:
    """
    Show parameter count per Transformer component.
    This helps understand where optimization efforts should focus.
    """
    breakdown = {
        "embeddings": (
            model.src_embed.weight.numel() +
            model.tgt_embed.weight.numel()
        ),
        "positional_encoding": model.positional_encoding.pe.numel(),
        "encoder_layers": sum(p.numel() for layer in model.encoder_layers for p in layer.parameters()),
        "decoder_layers": sum(p.numel() for layer in model.decoder_layers for p in layer.parameters()),
        "output_projection": model.output_proj.weight.numel() + model.output_proj.bias.numel(),
    }
    return breakdown


# Profile our model
memory_profile = estimate_memory_bytes(model)
param_breakdown = count_parameters_by_component(model)

print("=" * 60)
print("MEMORY PROFILE (Training)")
print("=" * 60)
print("\nMemory per component:")
for name, bytes_used in memory_profile.items():
    mb = bytes_used / (1024 ** 2)
    print(f"  {name:20s}: {mb:8.2f} MB")

print("\n" + "=" * 60)
print("PARAMETER BREAKDOWN (where computation focuses)")
print("=" * 60)
total_params = sum(param_breakdown.values())
for name, params in param_breakdown.items():
    pct = 100 * params / total_params
    print(f"  {name:20s}: {params:8,} ({pct:5.1f}%)")
print(f"  {'TOTAL':20s}: {total_params:8,}")

# Interview question: How would we optimize this?
print("\n" + "=" * 60)
print("OPTIMIZATION STRATEGIES FOR INFRA ENGINEERS")
print("=" * 60)
print("""
1. REDUCE ACTIVATIONS (biggest win for memory):
   - Gradient checkpointing: Trade compute for memory
   - Recompute some layers instead of storing activations
   - Can fit 2-3x larger batch size

2. REDUCE WEIGHTS:
   - Quantization (int8, fp16, bfloat16)
   - Reduces memory by 2-4x but hurts training quality slightly
   
3. REDUCE OPTIMIZER STATES:
   - Use lower-precision optimizer states (fp16)
   - SGD instead of Adam (1x params instead of 3x)
   - Clear tradeoff: faster convergence vs memory

4. SEQUENCE LENGTH:
   - O(L·d·T·B) means doubling seq_len doubles activation memory
   - Flash Attention helps but doesn't eliminate this
   - Real constraint for long-context training

Example: With gradient checkpointing, you can often fit:
- 2-3x larger batch size
- Or 2x longer sequences
- The cost: ~30% slower (recompute forward pass during backward)
""")


## 11. Interview notes: the major aspects of a Transformer

Here is a compact review list.

### 1) Embeddings
Map discrete token IDs into dense vectors.

### 2) Positional information
Attention alone is permutation-invariant, so the model needs order information.

### 3) Scaled dot-product attention
Computes a learned weighted mixture of values based on query-key similarity.

### 4) Multi-head attention
Lets the model learn multiple relation types in parallel.

### 5) Feed-forward sublayer
Applies nonlinear feature transformation independently at every position.

### 6) Residual connections + layer norm
Make deep stacking trainable and stable.

### 7) Masking
- source padding mask for encoder
- target padding mask for decoder
- causal mask for decoder self-attention
- source padding awareness in cross-attention

### 8) Teacher forcing during training
Use shifted gold targets as decoder input.

### 9) Autoregressive decoding during inference
Generate one token at a time and feed it back in.

### 10) Complexity
Full attention is typically **O(T^2)** in sequence length for both compute and attention-score memory.

### 11) Why GPUs matter
Attention and MLPs are mostly big tensor operations and matrix multiplies.

### 12) How this relates to modern LLMs
Most modern LLMs are **decoder-only**:
- remove the encoder
- remove decoder cross-attention
- keep masked self-attention
- add many more decoder blocks
- use autoregressive next-token prediction

### 13) KV cache intuition
During autoregressive decoder-only inference, recomputing keys/values for old tokens is wasteful.  
A KV cache stores them so generation can reuse old work instead of recomputing everything every step.

That is one of the most important systems ideas in LLM inference.

## 13. Distributed Training & Inference: The Infrastructure Multiplier

### Why Distributed?

- Single GPU max batch size: Usually 4-12 on a V100/A100 (with gradient checkpointing)
- You want batch size 64+ for good training dynamics
- Need to spread computation across multiple GPUs/TPUs
- Inference: 1 GPU can do ~100-500 tokens/sec; serving needs 10,000s tokens/sec

### Three Approaches (Concepts, Not Implementation)

#### 1. **Data Parallelism (DDP in PyTorch)**

**Idea:** Same model on each GPU, different batch data.

```
GPU 0: Forward/backward on batch 0 → compute gradients
GPU 1: Forward/backward on batch 1 → compute gradients
GPU 2: Forward/backward on batch 2 → compute gradients

All-reduce: Average gradients across GPUs → SGD step
```

**Pros:**
- Simple to implement
- Scales well (near-linear scaling for 2-8 GPUs)

**Cons:**
- Doesn't help if model doesn't fit on one GPU
- Communication overhead grows with model size

**Memory advantage:** None. You still need full model on each GPU.

#### 2. **Fully Sharded Data Parallelism (FSDP)**

**Idea:** Shard model weights across GPUs, gather during forward/backward.

```
GPU 0: Stores layer parameters for decoder blocks 0-3
GPU 1: Stores layer parameters for decoder blocks 4-7
GPU 2: Stores layer parameters for decoder blocks 8-11

Forward pass:
- GPU 0 gathers layer 4-7 weights from GPU 1, computes
- GPU 0 gathers layer 8-11 weights from GPU 2, computes
```

**Pros:**
- Can fit 100B+ models on 8 GPUs
- Automatic gradient checkpointing integration
- Near-linear scaling

**Cons:**
- Complex collective operations (all-gather, reduce-scatter)
- Communication can become bottleneck on slow interconnects
- Requires careful tuning

**Memory advantage:** ~1/num_gpus model size per GPU + same optimizer states.

#### 3. **Pipeline Parallelism**

**Idea:** Different layers on different GPUs, pass activations between.

```
GPU 0: Layers 0-6 → send activations to GPU 1
GPU 1: Layers 7-12 → send activations to GPU 2
GPU 2: Layers 13-24 → send logits to loss
```

**Pros:**
- Works for very large models (1T+)
- Can balance compute across GPUs

**Cons:**
- **Bubble**: GPUs are idle waiting for previous stages (20-40% overhead)
- Hard to debug
- Requires model reorganization

**When used:** Rarely for training, sometimes for inference.

### Inference Distribution Strategies

The infrastructure difference from training is **huge**:

**Training:** You want maximum throughput (batch size = 256+, overlapped compute)  
**Inference:** You want low latency (serve 1 query in <100ms) AND high throughput (1000s queries/sec)

**Common approaches:**
1. **Tensor Parallelism** (split large model across GPUs for single request)
2. **Batch Parallelism** (different requests on different GPUs)
3. **Pipeline Parallelism** (different model chunks)
4. **KV-cache distribution** (cache on different GPU, send activations)

### Key Interview Questions on Distributed Systems

**Q: If you double the number of GPUs, does your training speed double?**  
A: No. Ideally ~1.7-1.8x (not 2x) because of communication overhead.

**Q: Why is inference harder to distribute than training?**  
A: Training batches slow requests together. Inference is high-latency-sensitive.

**Q: If you use FSDP with 8 GPUs, do you need 8x less model weights?**  
A: No. Each GPU still has optimizer states (~3x params). You save on weights only, so roughly 3-4x less memory per GPU.

**Q: What happens if allreduce takes longer than compute?**  
A: Communication becomes the bottleneck. You need faster interconnects (NVLink, Infiniband).


## 14. Quantization and Mixed Precision: Inference Speedups

### The Accuracy-Speed Tradeoff

**Full precision (float32):**
- Weights: 4 bytes each
- Activations during compute: 4 bytes each
- Slowest, most accurate

**Mixed precision (float32 weights, float16 compute):**
- Store weights as float32
- Do matrix multiplies in float16 (2-3x faster on GPUs with Tensor Cores)
- Tiny accuracy loss

**Quantization (int8 weights):**
- Store weights as integers (1 byte instead of 4)
- 4x smaller model, 2-4x faster inference
- Noticeable accuracy loss but acceptable (1-2% top-1 drop)

### Why It Matters

Real-world LLM inference:
- Open source models: Almost always served in **int8 quantization** (throughput > accuracy)
- Enterprise: Mix of **mixed precision** (low latency req) and **int8** (throughput)
- Research: **float32** only (accuracy critical)

**Scaling to production:**  
A model that's 28GB in float32 becomes:
- 14GB with float16 (mixed precision)
- 7GB with int8 quantization

Can now fit on 1 GPU instead of 4.

### Interview Concepts (No Implementation)

**Q: How does quantization affect inference latency?**  
A: Weights/activations are smaller → faster memory transfer → higher compute throughput.  
Modern GPUs have specialized int8 operations (much faster than float32).

**Q: Why don't we train in int8?**  
A: Because quantization error creates noise in gradients during backprop.  
Training is sensitive to this noise (divergence or poor convergence).

**Q: What's the difference between int8 and int4 quantization?**  
A: int4 is more aggressive (16x smaller) but accuracy drop is bigger (~5%).  
Reserved for very throughput-critical deployments.


## 15. Real-World Infrastructure Patterns: What You'll Actually Build

This section is **not** in the original paper or toy model, but absolutely critical for infrastructure jobs.

### Training Pipeline Pattern

```
1. Data Loading (I/O bound)
   ├─ Read sequences from disk
   ├─ Tokenize
   └─ Batch into tensors

2. Forward Pass (Compute + Memory)
   ├─ Allocate activations
   ├─ Distributed forward (DDP/FSDP)
   └─ Loss computation

3. Backward Pass (Compute + Memory)
   ├─ All-reduce gradients (for DDP)
   └─ Update weights (optimizer)

4. Checkpointing (Storage)
   └─ Save model, optimizer state, training metrics

Bottlenecks: Data I/O (solve with prefetch), communication (solve with gradient compression)
```

### Inference Serving Pattern

```
1. Request arrives with prompt
   └─ Encode prompt to token IDs

2. Prefill phase (high throughput):
   ├─ Process entire prompt at once (batch parallelism)
   ├─ Build KV cache for all prompt tokens
   └─ Compute logits for first generated token

3. Decode phase (low latency):
   ├─ One token at a time (autoregressive)
   ├─ Append to KV cache
   ├─ Use KV cache for fast forward pass
   └─ Repeat until <eos> or max_tokens

4. Return generated tokens to user

Bottlenecks: KV cache memory (solve with eviction/compression), decode latency (solve with better GPU placement)
```

### Common Infrastructure Decisions

| Decision | Impact | Example |
|----------|--------|---------|
| **GPU type** | Compute & memory speed | A100 vs H100 vs RTX 4090 |
| **Batch size** | Throughput vs latency | Inference: batch 1, Training: batch 256 |
| **Sequence length** | Activation memory | 2048 vs 4096 vs 100k context |
| **Precision** | Speed, accuracy, memory | float32 vs float16 vs int8 |
| **Distributed strategy** | How to scale | Batch parallel vs tensor parallel |
| **Checkpoint frequency** | Recovery vs I/O overhead | Every 100 steps vs every 1000 |

### What Interview Questions Actually Sound Like

**Training engineer interview:**
- "We want to train a 70B model. We have 8 A100s. How would you distribute it?"
- "Our training loss plateaus after 2 epochs. How would you debug this?"
- "Current throughput is 500 tokens/sec. How do we get to 2000?"

**Inference engineer interview:**
- "We're serving Llama-2 70B. Latency is 500ms. Target is 100ms. What do you optimize first?"
- "KV cache is taking 80% of our GPU memory. How do we solve this?"
- "A single A100 serves 10 requests/sec. We need 1000 requests/sec. What's your plan?"

### The Hierarchy of Concerns

1. **Does it work?** (Implementation)
2. **Is it fast enough?** (Optimization - profiling, bottleneck analysis)
3. **Can we scale it?** (Distributed + resource management)
4. **Can we monitor it?** (Metrics, logging, alerting)
5. **Can we debug it when it breaks?** (Observability)

Infrastructure engineers spend most time on 2-5, not 1.


## 16. Infrastructure Engineer Interview Cheatsheet

### Quick Reference: Key Numbers

```
Model sizes:
- 7B model ≈ 28GB in float32, 14GB in float16, 7GB in int8
- Scales linearly: 70B ≈ 10x more memory

Batch sizes during training:
- Single GPU: 4-16 (with gradient checkpointing)
- 8-GPU DDP: 32-128
- Goal: Large enough for stable training, small enough to fit

Memory breakdown (training):
- Model weights: 30%
- Optimizer states (Adam): 40%
- Activations: 20%
- Gradients: 10%

Inference latency (rule of thumb):
- Prefill phase: ~100-300 ms per 1024 tokens
- Decode phase: ~50-100 ms per token (with KV cache)
- Without KV cache: 10-100x slower
```

### Three Levels of Understanding (For Interview Prep)

#### Level 1: "I understand the concepts"
- KV cache speeds up generation 10-100x
- FSDP lets you fit bigger models through sharding
- Quantization trades accuracy for speed
- **This is the baseline** for any infrastructure role

#### Level 2: "I can explain tradeoffs"
- "DDP scales well on fast interconnects but has communication overhead"
- "Gradient checkpointing saves memory at 30% compute cost"
- "int8 quantization reduces model size 4x but may hurt accuracy 1-2%"
- **This is what gets you hired for mid-level roles**

#### Level 3: "I've built systems with this"
- "I profiled our training and found compute-bound vs I/O-bound phases"
- "I implemented dynamic batching to improve inference throughput"
- "I tuned FSDP bucket sizes for our specific interconnect"
- **This is for senior/staff roles** (but you don't need this level for initial interviews)

### If You Get Asked...

**"What's the main bottleneck in Transformer inference?"**  
A: KV cache memory. Each token we generate needs to cache K, V for all past tokens.  
At sequence length 10k, that's huge. Optimization: KV cache eviction, quantization, or better hardware.

**"Why is my 70B model training so slow on 8 GPUs?"**  
A: Profile it first to find the bottleneck:
- Is communication taking >30% of time? (Use all-reduce overlap, gradient compression)
- Are GPU util <70%? (Increase batch size if memory allows)
- Is I/O slow? (Prefetch more batches, cache preprocessing)

**"We want to serve a 13B model to 10k requests/sec. What's your plan?"**  
A: Impossible on single GPU. Need:
1. Multiple GPU instances (tensor or batch parallelism)
2. KV cache optimization (quantized cache, eviction policy)
3. Continuous batching (don't waste idle GPU time waiting for full batch)
4. Request scheduling (prioritize short requests)

**"How do we make training 2x faster?"**  
A: Depends on current bottleneck (need to profile first):
- If compute-bound: Better hardware (V100→A100), mixed precision
- If communication-bound: Gradient accumulation, better interconnects
- If memory-bound: Gradient checkpointing, smaller batch size (seems backward but reduces swap)

### What Makes a Good Infrastructure Interview Answer

1. **Ask clarifying questions first** ("Are we training or serving? Single or multi-GPU?")
2. **Show you know the tradeoffs** (not "this is fastest" but "this is fastest for X constraints")
3. **Talk about measurement** ("I'd profile to find the bottleneck before optimizing")
4. **Give numbers** ("That's O(T²) memory, which becomes <30GB at 4k sequence length")
5. **Connect to real systems** ("Similar to how vLLM handles continuous batching")

### Quick Debugging Flowchart

```
System is slow?
├─ GPU utilization <50%?
│  ├─ Compute-bound? → Batch size up, mixed precision
│  └─ Communication-bound? → Reduce communication, FSDP tuning
├─ GPU memory OOM?
│  ├─ Training? → Gradient checkpointing, smaller batch
│  └─ Inference? → KV cache quantization, sequence truncation
└─ Correct (just verify with profiler)
```

### Resources That Appear in Real Interviews

- **vLLM** (continuous batching inference framework)
- **DeepSpeed** (distributed training optimization)
- **Megatron-LM** (tensor parallelism library)
- **Flash Attention** (optimized attention kernels)
- **Hugging Face Transformers** (standard library)

You won't implement these, but knowing what they do and why they matter is gold.

### Final Thoughts

Infrastructure engineering is about **understanding constraints**:
- Memory constraints → Quantization, sharding, checkpointing
- Latency constraints → KV cache, batching, GPU placement
- Throughput constraints → Parallelism, batch size, precision
- Cost constraints → Optimize the above for $ / token or $ / inference

The best candidates don't memorize numbers—they understand **how to measure, profile, and reason** about these constraints.


## 12. Interview-style exercises

### Exercise 2 — Shape tracing
Suppose:
- batch size = 2
- sequence length = 5
- `d_model = 32`
- `num_heads = 4`

What are the shapes of:
1. `Q`, `K`, `V` after splitting into heads?
2. the attention score tensor `QK^T`?
3. the merged attention output before the final output projection?

**Hint:**  
`head_dim = d_model / num_heads`

### Exercise 2 — Answer

- `head_dim = 32 / 4 = 8`

After projection and split:
1. `Q`, `K`, `V` shapes are **[2, 4, 5, 8]**
2. attention scores `QK^T` shape is **[2, 4, 5, 5]**
3. merged output shape is **[2, 5, 32]**

This is a very common interview tracing exercise.

### Exercise 3 — Why does the encoder not need a causal mask?

**Hint:**  
Think about what information is allowed to be visible on the source side.

### Exercise 3 — Answer

The encoder is reading the full source sequence.  
It is not generating left-to-right, so it is allowed to attend to all source positions.

The decoder is different:
- it is generating autoregressively
- it must not peek at future target tokens during self-attention

So the decoder needs a causal mask; the encoder does not.

### Exercise 4 — Training vs inference

During training we used teacher forcing.  
During inference we used greedy decoding.

What is the key behavioral difference between them, and why can inference be slower?

**Hint:**  
Think about parallelism across time steps.

### Exercise 4 — Answer

During teacher forcing, the whole target-input sequence is already available, so the decoder can process all positions in parallel.

During autoregressive inference, the next token is unknown until the previous token is produced.  
So decoding becomes step-by-step:
1. generate one token
2. append it
3. run the decoder again
4. repeat

That makes inference less parallel across time and often slower per generated token.

### Exercise 5 — Complexity and systems angle

Why does full attention become expensive for long sequences?

**Hint:**  
Focus on the size of the attention score matrix.

### Exercise 5 — Answer

For a sequence length `T`, each attention head builds score matrices of size roughly `T x T`.

That means:
- compute grows roughly as **O(T^2)**
- memory for attention scores also grows roughly as **O(T^2)**

This is one reason long-context serving is challenging: attention cost grows quickly with sequence length.

### Exercise 6 — Convert this notebook to a decoder-only GPT-style model

What would you remove or change?

**Hint:**  
Ask yourself which parts exist only because we have both a source sequence and a target sequence.

### Exercise 6 — Answer

To move toward a decoder-only GPT-style model, you would:

- remove the encoder stack
- remove decoder cross-attention
- keep decoder masked self-attention
- keep token embeddings and positional information
- train on next-token prediction
- decode autoregressively at inference time

That is one of the most important architecture transitions to understand for LLM interviews.

### Exercise 7 — Small coding challenge

Modify the toy task from **copy** to **reverse**.

Current behavior:
- source: `3 1 4 <eos>`
- target: `3 1 4 <eos>`

Desired behavior:
- source: `3 1 4 <eos>`
- target: `4 1 3 <eos>`

**Hint:**  
Only the data generation function needs to change.  
Look at how `digits` is used inside `make_copy_batch`.

### Exercise 7 — Answer

Inside the batch generator, change:

```python
tgt_input = [BOS_ID] + digits
tgt_output = digits + [EOS_ID]
```

to:

```python
reversed_digits = list(reversed(digits))
tgt_input = [BOS_ID] + reversed_digits
tgt_output = reversed_digits + [EOS_ID]
```

That gives you a slightly more interesting seq2seq task while keeping the model code the same.

## 13. Final practical advice for interviews

If you can explain the following clearly, you are in a strong place:

1. how embeddings and positional encoding enter the model  
2. how self-attention works, including the shapes  
3. why multi-head attention is useful  
4. why the decoder needs a causal mask  
5. how teacher forcing differs from inference  
6. why attention is expensive for long sequences  
7. why GPUs are helpful for Transformers  
8. how the full encoder-decoder model differs from decoder-only LLMs  
9. why KV caching matters during autoregressive inference

That combination is useful across:
- model training interviews
- LLM inference interviews
- ML systems / infra interviews

## References for further reading

- *Attention Is All You Need*  
- PyTorch documentation for `torch.nn.functional.scaled_dot_product_attention`  
- PyTorch documentation for `nn.MultiheadAttention`  
- PyTorch documentation for `nn.TransformerEncoderLayer`  
- PyTorch Transformer building-block tutorials

For interview prep, a very effective next step is:
1. understand this notebook deeply  
2. switch the toy task to reverse or translation-like mapping  
3. increase model size and train steps on Colab GPU  
4. then move on to decoder-only and KV-cache-oriented notebooks